In [ ]:
import sys
from contextlib import redirect_stdout
from pathlib import Path
import io

sys.path.insert(0, str(Path('../..').resolve()))

In [ ]:
import pandas as pd

df = pd.read_parquet('../ssi_all_features_30_days.parquet')
df.shape

In [ ]:
before_delivery_med_features = df.loc[df['description'] == 'before_delivery_medadmin_ingredient', 'feature_name'].unique().tolist()
after_delivery_med_features = df.loc[df['description'] == 'after_delivery_medadmin_ingredient', 'feature_name'].unique().tolist()
or_timings_features = df.loc[df['description'] == 'or_timings', 'feature_name'].unique().tolist()
len(before_delivery_med_features), len(after_delivery_med_features), len(or_timings_features)

In [ ]:
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
df_pivoted.columns.name = ''
df_pivoted.shape

In [ ]:
from ssi.preprocess import preprocess

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_preprocessed = preprocess(df_pivoted, defer_imputation=True)

In [ ]:
df_pivoted_preprocessed.isna().sum().sort_values(ascending=False)

In [ ]:
from ssi.feature_filter import feature_filter

excluded = ['target_variable', 'birthid', 'mother_patid']
X = df_pivoted_preprocessed.drop(columns=excluded)
y = df_pivoted_preprocessed['target_variable']
mother_patid = df_pivoted_preprocessed['mother_patid']

# Use the filtered schema to define predictors. Imputation runs inside CV.
with redirect_stdout(io.StringIO()):
    df_pivoted_filtered = feature_filter(X.copy(), skip_medadmin_ingredient=True)

In [ ]:
df_pivoted_filtered.isna().sum().sort_values(ascending=False)

In [ ]:
before_delivery_med_features = [x for x in before_delivery_med_features if x in df_pivoted_filtered.columns]
after_delivery_med_features = [x for x in after_delivery_med_features if x in df_pivoted_filtered.columns]
or_timings_features = [x for x in df_pivoted_filtered.columns if x.startswith('or_timings')]
len(before_delivery_med_features), len(after_delivery_med_features), len(or_timings_features)

In [ ]:
med_features = [x for x in df_pivoted_filtered.columns if 'medadmin_ingredient' in x]
len(med_features)  # this has more since it has _isna features in it

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from src.clinical_imputer import ClinicalImputer
from src.model_training import nested_cv_evaluation_feature_selection

param_grid = {'clf__min_samples_split': [20, 50, 70, 100, 200, 500]}
cv_settings = dict(groups=mother_patid, outer_splits=5, inner_splits=5, random_state=range(10))

def rf_pipeline(columns):
    return Pipeline([
        ('clinical', ClinicalImputer(columns=columns)),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
    ])

### 1 pcornet features only

In [ ]:
pcornet_features = [x for x in df_pivoted_filtered.columns if x not in or_timings_features]
len(pcornet_features)

### 2. med features only using start time

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
        if x.endswith('earliest_minute'):
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)

In [ ]:
model_features = pcornet_features_subset
print(f'Predictors: {len(model_features)}')
metrics_2 = nested_cv_evaluation_feature_selection(
    X, y, rf_pipeline(model_features), param_grid, **cv_settings,
)
metrics_2['number_of_features'] = len(model_features)

### 3. pcornet subset without start time and or timings

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)
print(len(pcornet_features_subset))

In [ ]:
model_features = pcornet_features_subset + or_timings_features
print(f'Predictors: {len(model_features)}')
metrics_3 = nested_cv_evaluation_feature_selection(
    X, y, rf_pipeline(model_features), param_grid, **cv_settings,
)
metrics_3['number_of_features'] = len(model_features)

### 4 only has med???

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)

model_features = pcornet_features_subset
print(f'Predictors: {len(model_features)}')
metrics_4 = nested_cv_evaluation_feature_selection(
    X, y, rf_pipeline(model_features), param_grid, **cv_settings,
)
metrics_4['number_of_features'] = len(model_features)

### 5, earliest + or timings

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
        if x.endswith('earliest_minute'):
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)

model_features = pcornet_features_subset + or_timings_features
print(f'Predictors: {len(model_features)}')
metrics_5 = nested_cv_evaluation_feature_selection(
    X, y, rf_pipeline(model_features), param_grid, **cv_settings,
)
metrics_5['number_of_features'] = len(model_features)

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

tmp1 = metrics_4
tmp1['test'] = 'Baseline EHR features'

tmp2 = metrics_3
tmp2['test'] = 'Baseline EHR + \nintraoperative timing features'

tmp3 = metrics_2
tmp3['test'] = 'Baseline EHR + \nmedication administration timing features'

tmp4 = metrics_5
tmp4['test'] = 'all features'

tmp = pd.concat([tmp1, tmp2, tmp3, tmp4], axis=0)
tmp.columns.name = ''

sns.barplot(tmp, x='roc_auc', y='test', errorbar=None)
plt.xlim([0.60, 0.72])
plt.ylabel('')
results_dir = Path('results/timing_30_days_revision')
results_dir.mkdir(parents=True, exist_ok=True)
tmp.to_csv(results_dir / 'outer_fold_metrics.csv', index=False)
tmp.groupby('test')[['roc_auc', 'ap', 'sens_80']].mean().to_csv(results_dir / 'performance_summary.csv')
plt.savefig(results_dir / 'supplementary_figure_1.png', bbox_inches='tight', dpi=200)

import json
from importlib.metadata import version

run_config = {
    'repeat_seeds': list(cv_settings['random_state']),
    'outer_splits': cv_settings['outer_splits'],
    'inner_splits': cv_settings['inner_splits'],
    'rf_grid': param_grid,
    'versions': {p: version(p) for p in ('numpy', 'pandas', 'scikit-learn')},
}
(results_dir / 'run_config.json').write_text(json.dumps(run_config, indent=2) + '\n')